# Phase 5c — ASB scorecard + Cross/PR strategy tune

Param-driven fit for `ins` / `css` / `pr` / `cross`, then strategy compare. Primary UI after fit: Streamlit workbench.

```bash
uv run python -m streamlit run apps/scorecard_workbench.py
```

§6 = one-product offline as-if smoke. §12 = closed-loop + as-if export for the Profit tab.


## §0 — Params (tune here)


In [ ]:
from pathlib import Path
import json
import pickle
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd
import yaml
from IPython.display import HTML, display

from credit_scoring.scorecard.params import artifact_registry, load_product_scorecard_params

ROOT = Path("..").resolve()
DATA = ROOT / "data"
MODELS = DATA / "06_models"
REPORTING = DATA / "08_reporting"
REPORTING.mkdir(parents=True, exist_ok=True)

# --- Tunables ---
PRODUCT = "css"          # ins | css | pr | cross
SAVE_VERSION = "v1"
SAVE = True             # flip True in §7 / §13 when satisfied
MANUAL_OVERRIDE = None   # e.g. ["act_age_WOE", ...] or None
WINNER_ROW = 0
RUN_COMBO_SEARCH = True

with open(ROOT / "conf" / "base" / "parameters.yml") as f:
    _yml = yaml.safe_load(f)

# Per-product scorecard params from YAML (binning, QC, prefixes) — edit parameters.yml
SCORECARD_PARAMS = load_product_scorecard_params(_yml, PRODUCT)

PROFIT_PARAMS = deepcopy(_yml["profit"])
# Decision cutoffs from YAML (not from calibration JSON). Edit here or in parameters.yml, then §12.
STRATEGY_CUTOFFS = dict(PROFIT_PARAMS.get("cutoffs") or {})

print("PRODUCT            = ", PRODUCT)
print("Target             = ", SCORECARD_PARAMS["target"])
print("Binning settings   = ", {k: SCORECARD_PARAMS[k] for k in ("ncategories_int", "minimum_share_int", "ncategories_nom")})
print("QC settings        = ", {k: SCORECARD_PARAMS[k] for k in ("max_bad_rate_swing", "min_static_br_gap", "min_period_br_gap", "soft_drop_qc_fails")})
print("Allowed refixes    = ", SCORECARD_PARAMS.get("prefixes"))
print("Blocked prefixes   = ", SCORECARD_PARAMS.get("blocked"))
print("STRATEGY_CUTOFFS   = ", STRATEGY_CUTOFFS)
print("SAVE               = ", SAVE)
print("Version            = ", SAVE_VERSION)
print("(calib JSON = score → PD only; cutoffs = profit.cutoffs / STRATEGY_CUTOFFS)")

_registry = artifact_registry(PROFIT_PARAMS, models_dir=MODELS, root=ROOT)
display(pd.DataFrame(_registry))


## §1 — Load ABT + decisions (+ cross labels if present)


In [ ]:
from credit_scoring.simulation.cross_labels import attach_cross_labels, cross_label_summary

abt_path = DATA / "04_feature" / "abt_app.parquet"
cross_path = DATA / "04_feature" / "abt_app_cross.parquet"
dec = pd.read_parquet(DATA / "04_feature" / "decisions.parquet")

if cross_path.exists():
    abt = pd.read_parquet(cross_path)
    print("Loaded labeled ABT:", cross_path)
else:
    abt = pd.read_parquet(abt_path)
    if "decision" not in abt.columns:
        abt = abt.merge(dec[["aid", "decision", "decline_reason"]], on="aid", how="left")
    print("Loaded base ABT; run §8 to attach cross labels")

print(abt.shape, "products=", abt["product"].value_counts().to_dict() if "product" in abt else None)


## §2 — Binning + WOE + prescreen + QC

Keep `app*`/`act*`, block `agr*`/`ags*` from the scorecard (agr still feeds the bad-customer rule). Soft-drop QC fails when `soft_drop_qc_fails`.


In [ ]:
import importlib
import credit_scoring.scorecard.selection_asb as _sel_asb
import credit_scoring.scorecard.stability as _stability
import credit_scoring.scorecard.tune as _tune
importlib.reload(_sel_asb)
importlib.reload(_stability)
importlib.reload(_tune)
from credit_scoring.scorecard.tune import prepare_scorecard_context

ctx = prepare_scorecard_context(abt, PRODUCT, SCORECARD_PARAMS)
print(
    f"feats={len(ctx['feats'])} maps={len(ctx['maps'])} "
    f"keep={len(ctx['woe_cols'])} train={len(ctx['train_w'])} valid={len(ctx['valid_w'])}"
)
assert "qc_table" in ctx, (
    "prepare_scorecard_context missing qc_table — restart kernel / reinstall editable package"
)
print("QC safe=", int(ctx["qc_table"]["safe_to_include"].sum()), "/", len(ctx["qc_table"]))
if len(ctx["woe_cols"]) < min(SCORECARD_PARAMS.get("number_features", (5,))):
    print(
        f"WARNING: only {len(ctx['woe_cols'])} vars left after QC soft-drop; "
        "combo search needs ≥ min(number_features). "
        "Relax scorecard.css QC, set soft_drop_qc_fails=False, or MANUAL_OVERRIDE."
    )
display(ctx["qc_table"])
display(ctx["uni"].head(12))
display(ctx["big_scorecard"].head(10))


## §3 — RFE + Model_list combinatorial search


In [ ]:
from credit_scoring.scorecard.tune import run_variable_search, fit_selected_model

selected = None
Model_list = subModel_list = pd.DataFrame()
best = None

if RUN_COMBO_SEARCH:
    try:
        Model_list, subModel_list, best = run_variable_search(
            ctx,
            number_vars=SCORECARD_PARAMS["number_vars"],
            number_features=SCORECARD_PARAMS["number_features"],
        )
        print("Model_list", Model_list.shape, "subModel_list", subModel_list.shape)
        display(subModel_list.head(15) if len(subModel_list) else Model_list.head(15))
    except Exception as exc:
        print(f"Combo search failed ({type(exc).__name__}: {exc})")
        print("Falling back — tighten QC, set MANUAL_OVERRIDE, or set soft_drop_qc_fails=False")
else:
    print("Skipped combo search — use MANUAL_OVERRIDE")

if MANUAL_OVERRIDE:
    selected = list(MANUAL_OVERRIDE)
elif len(subModel_list):
    row = subModel_list.iloc[min(WINNER_ROW, len(subModel_list) - 1)]
    selected = row["Variables"].split(",")
elif best is not None:
    selected = best["feature_cols"]
elif len(ctx["woe_cols"]):
    selected = list(ctx["woe_cols"])[: max(SCORECARD_PARAMS["number_features"])]
    print("WARNING: fallback to surviving woe_cols (search empty / too few vars after QC)")
else:
    selected = ctx["uni"].head(5)["woe"].tolist()
    print("WARNING: fallback to top-5 univariate WOE")

if not selected:
    raise RuntimeError(
        "No features selected. QC soft-drop may have emptied the pool "
        f"(keep={len(ctx.get('woe_cols', []))}). Set MANUAL_OVERRIDE or relax QC."
    )

print("selected:", selected)
fit = fit_selected_model(
    ctx,
    selected,
    factor=SCORECARD_PARAMS["factor"],
    offset=SCORECARD_PARAMS["offset"],
)
print("metrics:", fit["metrics"])
display(fit["model_report"]["Effects"])
display(fit["model_report"]["Main_measures"])


## §4 — Workbench bundle

Writes `data/08_reporting/workbench_bundle/{PRODUCT}/` for Streamlit. Optional: `python scripts/export_asif_for_workbench.py` for the Profit tab.


In [ ]:
from credit_scoring.profit.cutoff_explore import save_workbench_product_bundle
from credit_scoring.scorecard.reports import render_scorecard_html

# Primary UI: Streamlit workbench (dossier + live cutoffs). HTML is an optional snapshot.
report_path = REPORTING / f"scorecard_report_{PRODUCT}.html"
profit_one = None  # filled by §6 (one-product as-if)
profit_all = None
_concl = REPORTING / "strategy_conclusion.json"
if _concl.exists():
    with open(_concl) as f:
        _sc = json.load(f)
    _block = _sc.get("midband") or _sc.get("pd_only") or _sc
    profit_all = {
        **{k: _block[k] for k in (
            "total_profit", "n_accept", "ar_ins", "ar_css", "bad_rate_ins", "bad_rate_css"
        ) if k in _block},
        "by_product": _block.get("by_product"),
        "reference": _sc.get("reference"),
        "beats_reference": (
            (_block.get("total_profit") > _sc["reference"])
            if _block.get("total_profit") is not None and _sc.get("reference") is not None
            else None
        ),
        "best_strategy": "midband" if _sc.get("midband") else "pd_only",
    }

bundle_dir = save_workbench_product_bundle(
    PRODUCT,
    {
        "model_package": fit["model_package"],
        "points_table": fit["points_table"],
        "calibration": fit["calibration"],
        "cal_table": fit["cal_table"],
        "gini_time": fit["gini_time"],
        "qc_table": ctx["qc_table"],
        "big_scorecard": ctx["big_scorecard"],
        "model_report": fit["model_report"],
        "variable_report": fit.get("variable_report"),
        "scored_train": fit["scored_train"],
        "scored_valid": fit["scored_valid"],
    },
    bundle_dir=REPORTING / "workbench_bundle",
)
print("Workbench bundle →", bundle_dir)
print("Run: python -m streamlit run apps/scorecard_workbench.py")

html = render_scorecard_html(
    product=PRODUCT,
    model_package=fit["model_package"],
    points_table=fit["points_table"],
    big_scorecard=ctx["big_scorecard"],
    model_report=fit["model_report"],
    gini_time=fit["gini_time"],
    calibration=fit["calibration"],
    cal_table=fit["cal_table"],
    train_scores=fit["scored_train"]["score"],
    valid_scores=fit["scored_valid"]["score"],
    qc_table=ctx["qc_table"],
    profit_one=profit_one,
    profit_all=profit_all,
    variable_report=fit.get("variable_report"),
    scored_for_roc=fit["scored_valid"],
    output_path=report_path,
)
print("Optional HTML snapshot →", report_path)
display(fit["gini_time"].tail(8))


## §5 — Review gate

Inspect `qc_table` / `subModel_list`. Change `WINNER_ROW` or `MANUAL_OVERRIDE` in §0 and re-run §3–§4.


In [ ]:
print("PRODUCT", PRODUCT)
print("features", selected)
print("gini_train/valid", fit["metrics"]["gini_train"], fit["metrics"]["gini_valid"])
print("HTML:", report_path)


## §6 — One-product PD profit smoke

Offline as-if P&L for the current `PRODUCT` (`ins`/`css` only) using the in-memory fit. For portfolio closed-loop, use §12.


In [ ]:
from credit_scoring.profit.pnl import compute_pnl_table, filter_profit_window
from credit_scoring.profit.scoring import score_product_slice

profit_one = None
if PRODUCT not in ("ins", "css"):
    print(f"PRODUCT={PRODUCT!r}: §6 P&L smoke is for ins/css app PD only. Re-run HTML without profit_one.")
else:
    abt_prod = abt.loc[abt["product"].astype(str).str.lower().eq(PRODUCT)].copy()
    # Keep default12 if present on abt; else merge from decisions path is already on abt_app
    scored_one = score_product_slice(
        abt_prod,
        fit["model_package"],
        fit["points_table"],
        fit["calibration"].get("params", fit["calibration"]),
    )
    scored_pnl = compute_pnl_table(scored_one, PROFIT_PARAMS["economics"])
    scored_win = filter_profit_window(
        scored_pnl, PROFIT_PARAMS["window_start"], PROFIT_PARAMS["window_end"]
    )
    cutoff_key = "pd_css" if PRODUCT == "css" else "pd_ins_high"
    cutoff = float(STRATEGY_CUTOFFS.get(cutoff_key, PROFIT_PARAMS.get("cutoffs", {}).get(cutoff_key)))
    accepted = scored_win.loc[scored_win["pd"] <= cutoff]
    n_apps = len(scored_win)
    n_accept = len(accepted)
    total_profit = float(accepted["profit"].sum()) if n_accept else 0.0
    ar = (n_accept / n_apps) if n_apps else float("nan")
    bad_rate = float(accepted["default12"].mean()) if n_accept and "default12" in accepted.columns else float("nan")
    reference = float(PROFIT_PARAMS.get("reference", float("nan")))
    profit_one = {
        "product": PRODUCT,
        "total_profit": total_profit,
        "n_accept": int(n_accept),
        "ar": ar,
        "bad_rate": bad_rate,
        "cutoff": cutoff,
        "reference": reference,
        "beats_reference": bool(total_profit > reference) if pd.notna(reference) else None,
        "note": "offline as-if on static abt_app; not closed-loop",
    }
    print(
        f"§6 one-product ({PRODUCT}): profit={total_profit:,.0f} n_accept={n_accept} "
        f"ar={ar:.3f} cutoff={cutoff:.4f} beats_ref={profit_one['beats_reference']}"
    )

html = render_scorecard_html(
    product=PRODUCT,
    model_package=fit["model_package"],
    points_table=fit["points_table"],
    big_scorecard=ctx["big_scorecard"],
    model_report=fit["model_report"],
    gini_time=fit["gini_time"],
    calibration=fit["calibration"],
    cal_table=fit["cal_table"],
    train_scores=fit["scored_train"]["score"],
    valid_scores=fit["scored_valid"]["score"],
    qc_table=ctx["qc_table"],
    profit_one=profit_one,
    profit_all=profit_all,
    output_path=report_path,
)
print("Updated HTML profit § one-product →", report_path)
display(HTML(f'<p><a href="{report_path}" target="_blank">Open HTML report</a></p>'))


## §7 — Save application PD / PR / Cross bundle


In [ ]:
from credit_scoring.scorecard.freeze import save_model_bundle

if SAVE:
    paths = save_model_bundle(
        product=PRODUCT,
        version=SAVE_VERSION,
        model_package=fit["model_package"],
        points_table=fit["points_table"],
        calibration=fit["calibration"],
        output_dir=MODELS,
        decision_log={
            "product": PRODUCT,
            "features": selected,
            "params": {k: SCORECARD_PARAMS[k] for k in (
                "ncategories_int", "minimum_share_int", "ncategories_nom",
                "train_end_period", "valid_start_period", "target",
                "max_bad_rate_swing", "min_static_br_gap", "min_period_br_gap",
            ) if k in SCORECARD_PARAMS},
            "metrics": fit["metrics"],
        },
    )
    print("Saved:")
    for k, v in paths.items():
        print(f"  {k}: {v}")
    print("Next: point profit.artifacts in parameters.yml at these paths, then flip PRODUCT.")
else:
    print("SAVE=False — flip to True when report + profit look good")


## §8 — Cross / PR labels

Attaches `cross_response`, `cross_aid`, `default_cross12`. Writes `data/04_feature/abt_app_cross.parquet`.


In [ ]:
abt_base = pd.read_parquet(DATA / "04_feature" / "abt_app.parquet")
if "decision" not in abt_base.columns:
    abt_base = abt_base.merge(dec[["aid", "decision", "decline_reason"]], on="aid", how="left")

abt_labeled = attach_cross_labels(abt_base, response_n_months=6)
summary = cross_label_summary(abt_labeled)
print(json.dumps(summary, indent=2))
assert summary["has_cross_response"] and summary["n_responders"] > 0

abt_labeled.to_parquet(cross_path, index=False)
abt = abt_labeled
print("Wrote", cross_path)


## §9–§10 — PR and Cross PD

Set `PRODUCT="pr"` then `"cross"` in §0 and re-run §2–§7. Mid-band keeps borderline INS when response propensity is high and cross PD is low enough.


In [ ]:
print('''Workflow reminder:
  PRODUCT="pr"   → SAVE_VERSION, SAVE=True after HTML OK
  PRODUCT="cross" → same
Artifacts:
  pr_css_{version}.pkl + points_table_pr_css_* + calibration_params_pr_css_*
  cross_pd_css_{version}.pkl + points_table_cross_pd_css_* + ...
Promote paths into parameters.yml profit.artifacts yourself after save.
''')
print("Current PRODUCT=", PRODUCT)
display(pd.DataFrame(artifact_registry(PROFIT_PARAMS, models_dir=MODELS, root=ROOT)))


## §11 — Strategy cutoffs (mid-band)

Edit `STRATEGY_CUTOFFS`, then run §12. Raise `pd_ins_high` above `pd_ins_low` before mid-band / PR / Cross can fire.


In [ ]:
# Loaded from profit.cutoffs in §0 — edit values then re-run §12
# STRATEGY_CUTOFFS = {'pd_css': 0.32, 'pd_ins_high': 0.0218, 'pd_ins_low': 0.01, 'pr_min': 0.028, 'cross_pd_max': 0.2724}


## §12 — Closed-loop: PD-only vs mid-band Strategy


In [ ]:
from credit_scoring.profit.cutoff_explore import (
    build_asif_scored_frame,
    export_asif_scored,
    save_workbench_product_bundle,
)
from credit_scoring.profit.pnl import compute_pnl_table, filter_profit_window
from credit_scoring.profit.resim import run_closed_loop_resim
from credit_scoring.profit.rules import evaluate_strategy, rules_from_params
from credit_scoring.profit.scoring import score_abt_application
from credit_scoring.scorecard.reports import render_scorecard_html

def _load_bundle(package_path, points_path, calib_path):
    with open(package_path, "rb") as f:
        pkg = pickle.load(f)
    pts = pd.read_parquet(points_path)
    with open(calib_path) as f:
        cal = json.load(f)
    return pkg, pts, cal

def load_artifacts(profit_params, root=ROOT):
    packages, points, cals = {}, {}, {}
    for product, paths in profit_params.get("artifacts", {}).items():
        p = root / paths["package"]
        if not p.exists():
            print("missing", p)
            continue
        packages[product], points[product], cals[product] = _load_bundle(
            p, root / paths["points"], root / paths["calib"]
        )
    return packages, points, cals

def _serialize_eval(ev):
    out = {k: ev[k] for k in (
        "total_profit", "n_accept", "ar_ins", "ar_css", "bad_rate_ins", "bad_rate_css"
    ) if k in ev}
    bp = ev.get("by_product")
    if isinstance(bp, pd.DataFrame) and len(bp):
        out["by_product"] = bp.to_dict(orient="records")
    return out

pp = deepcopy(PROFIT_PARAMS)
pp["cutoffs"] = {**pp.get("cutoffs", {}), **STRATEGY_CUTOFFS}
behavioral_params = _yml["behavioral"]
sim_params = _yml["simulation"]

# Promote SAVE_VERSION / pr / cross if present on disk
for prod in ("ins", "css"):
    pkg = MODELS / f"pd_{prod}_{SAVE_VERSION}.pkl"
    if pkg.exists():
        pp.setdefault("artifacts", {})
        pp["artifacts"][prod] = {
            "package": str(pkg.relative_to(ROOT)),
            "points": str((MODELS / f"points_table_{prod}_{SAVE_VERSION}.parquet").relative_to(ROOT)),
            "calib": str((MODELS / f"calibration_params_{prod}_{SAVE_VERSION}.json").relative_to(ROOT)),
        }
for prod, stem in (("pr", f"pr_css_{SAVE_VERSION}"), ("cross", f"cross_pd_css_{SAVE_VERSION}")):
    pkg = MODELS / f"{stem}.pkl"
    if pkg.exists():
        pp.setdefault("artifacts", {})
        pp["artifacts"][prod] = {
            "package": str(pkg.relative_to(ROOT)),
            "points": str((MODELS / f"points_table_{stem}.parquet").relative_to(ROOT)),
            "calib": str((MODELS / f"calibration_params_{stem}.json").relative_to(ROOT)),
        }

packages, points_tables, calibrations = load_artifacts(pp)
print("loaded artifacts:", sorted(packages))
print("using cutoffs:", pp["cutoffs"])

# As-if scored frame for Streamlit Profit tab (fast cutoff exploration)
secondary = {
    k: {"package": packages[k], "points": points_tables[k], "calib": calibrations[k]}
    for k in ("pr", "cross") if k in packages
}
if "ins" in packages and "css" in packages:
    asif = build_asif_scored_frame(
        abt,
        {k: packages[k] for k in ("ins", "css")},
        {k: points_tables[k] for k in ("ins", "css")},
        {k: calibrations[k] for k in ("ins", "css")},
        pp["economics"],
        secondary=secondary or None,
    )
    export_asif_scored(
        asif,
        {
            "window_start": pp["window_start"],
            "window_end": pp["window_end"],
            "burn_in_before": pp.get("burn_in_before"),
            "reference": pp.get("reference"),
            "artifacts": pp.get("artifacts"),
            "secondary_keys": list(secondary),
            "n_rows": int(len(asif)),
        },
        parquet_path=REPORTING / "asif_scored_for_tuner.parquet",
        meta_path=REPORTING / "asif_scored_for_tuner_meta.json",
    )
    print("Exported as-if scored →", REPORTING / "asif_scored_for_tuner.parquet")

production = pd.read_parquet(DATA / "02_intermediate" / "production.parquet")
transactions = pd.read_parquet(DATA / "02_intermediate" / "transactions.parquet")
default_df = pd.read_parquet(DATA / "02_intermediate" / "default.parquet")

rules_st = rules_from_params(pp)
rules_pd = deepcopy(rules_st)
rules_pd["cutoffs"] = {k: rules_pd["cutoffs"][k] for k in ("pd_css", "pd_ins_high") if k in rules_pd["cutoffs"]}

def _run(rules, label, use_secondary=True):
    packs = {k: packages[k] for k in packages if k in ("ins", "css") or use_secondary}
    pts = {k: points_tables[k] for k in packs}
    cals = {k: calibrations[k] for k in packs}
    abt_r, dec_r = run_closed_loop_resim(
        production,
        transactions,
        default_df,
        behavioral_params,
        sim_params,
        pp,
        packs,
        pts,
        cals,
        start_period=pp["window_start"],
        end_period=pp["window_end"],
        verbose=False,
        rules_override=rules,
    )
    scored = score_abt_application(
        abt_r,
        {k: packs[k] for k in ("ins", "css") if k in packs},
        {k: pts[k] for k in ("ins", "css") if k in packs},
        {k: cals[k] for k in ("ins", "css") if k in packs},
        secondary={
            k: {"package": packs[k], "points": pts[k], "calib": cals[k]}
            for k in ("pr", "cross") if k in packs and use_secondary
        } or None,
    )
    scored_pnl = compute_pnl_table(scored, pp["economics"])
    scored_window = filter_profit_window(scored_pnl, pp["window_start"], pp["window_end"])
    ev = evaluate_strategy(scored_window, dec_r, pp["window_start"], pp["window_end"])
    print(
        f"{label}: profit={ev['total_profit']:,.0f} n={ev['n_accept']} "
        f"ar_ins={ev['ar_ins']:.3f} ar_css={ev['ar_css']:.3f} "
        f"beats={ev['total_profit'] > pp['reference']}"
    )
    return ev

assert "ins" in packages and "css" in packages, "Need ins+css packages before §12"
print("Running closed-loop (several minutes)...")
ev_pd = _run(rules_pd, "PD-only", use_secondary=False)
ev_st = _run(rules_st, "St1-midband", use_secondary=True) if ("pr" in packages and "cross" in packages) else None
if ev_st is None:
    print("PR/Cross not frozen yet — mid-band skipped")

conclusion = {
    "pd_only": _serialize_eval(ev_pd),
    "midband": _serialize_eval(ev_st) if ev_st else None,
    "reference": pp["reference"],
    "cutoffs": pp["cutoffs"],
    "artifacts": pp.get("artifacts"),
}
with open(REPORTING / "strategy_conclusion.json", "w") as f:
    json.dump(conclusion, f, indent=2)
print("Wrote", REPORTING / "strategy_conclusion.json")

_block = conclusion.get("midband") or conclusion["pd_only"]
profit_all = {
    **{k: _block[k] for k in (
        "total_profit", "n_accept", "ar_ins", "ar_css", "bad_rate_ins", "bad_rate_css"
    ) if k in _block},
    "by_product": _block.get("by_product"),
    "reference": conclusion["reference"],
    "beats_reference": bool(_block["total_profit"] > conclusion["reference"]),
    "best_strategy": "midband" if conclusion.get("midband") else "pd_only",
}
print("Prefer Streamlit Profit tab for cutoff search:", "python -m streamlit run apps/scorecard_workbench.py")
if "fit" in dir() and "report_path" in dir():
    save_workbench_product_bundle(
        PRODUCT,
        {
            "model_package": fit["model_package"],
            "points_table": fit["points_table"],
            "calibration": fit["calibration"],
            "cal_table": fit["cal_table"],
            "gini_time": fit["gini_time"],
            "qc_table": ctx["qc_table"],
            "big_scorecard": ctx["big_scorecard"],
            "model_report": fit["model_report"],
            "variable_report": fit.get("variable_report"),
            "scored_train": fit["scored_train"],
            "scored_valid": fit["scored_valid"],
        },
        bundle_dir=REPORTING / "workbench_bundle",
    )
    html = render_scorecard_html(
        product=PRODUCT,
        model_package=fit["model_package"],
        points_table=fit["points_table"],
        big_scorecard=ctx["big_scorecard"],
        model_report=fit["model_report"],
        gini_time=fit["gini_time"],
        calibration=fit["calibration"],
        cal_table=fit["cal_table"],
        train_scores=fit["scored_train"]["score"],
        valid_scores=fit["scored_valid"]["score"],
        qc_table=ctx["qc_table"],
        profit_one=profit_one if "profit_one" in dir() else None,
        profit_all=profit_all,
        variable_report=fit.get("variable_report"),
        scored_for_roc=fit["scored_valid"],
        output_path=report_path,
    )
    print("Updated optional HTML snapshot →", report_path)
else:
    print("Skip dossier refresh (no in-memory fit — re-run §4).")


## §13 — Promote artifacts

§12 writes `strategy_conclusion.json` but does not edit YAML. With `SAVE=True`, freeze bundles, then update `profit.artifacts` / `profit.cutoffs` in `conf/base/parameters.yml`.


In [ ]:
# After both PD + PR/Cross saved and §12 looks good:
# 1. Update conf/base/parameters.yml profit.artifacts to SAVE_VERSION / pr / cross paths
# 2. Set profit.cutoffs to STRATEGY_CUTOFFS
# 3. Re-run notebooks/04_profit.ipynb (Part B) for the official claim
print("STRATEGY_CUTOFFS =", STRATEGY_CUTOFFS)
print("SAVE_VERSION =", SAVE_VERSION)
display(pd.DataFrame(artifact_registry(PROFIT_PARAMS, models_dir=MODELS, root=ROOT)))
print("Promote: SAVE=True → freeze → edit conf/base/parameters.yml profit.artifacts / cutoffs.")
